In [1]:
import numpy as np
import pandas as pd
import ml_utils as mu

data = np.load("../results/ml/morgan.npz", allow_pickle=True)
X_morgan, ids = data["X"], data["ids"]
ifp = pd.read_csv("../results/ml/ifp.csv", index_col=0)
comp = pd.read_csv("../results/ml/compounds.csv")

assert list(comp["id"]) == list(ids) == list(ifp.index), "feature files are out of order"

y = comp["label"].to_numpy()
groups = comp["scaffold"].to_numpy()
X_ifp = ifp.to_numpy()

print("Morgan:", X_morgan.shape, " IFP:", X_ifp.shape, " actives:", int(y.sum()))

Morgan: (1219, 2048)  IFP: (1219, 115)  actives: 40


In [2]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=mu.SEED)
for k, (train_idx, test_idx) in enumerate(cv.split(X_morgan, y, groups), start=1):
    print(f"fold {k}:  train {len(train_idx):4d} ({int(y[train_idx].sum())} actives)"
          f"   test {len(test_idx):4d} ({int(y[test_idx].sum())} actives)")

fold 1:  train  978 (33 actives)   test  241 (7 actives)
fold 2:  train  964 (27 actives)   test  255 (13 actives)
fold 3:  train  978 (34 actives)   test  241 (6 actives)
fold 4:  train  978 (33 actives)   test  241 (7 actives)
fold 5:  train  978 (33 actives)   test  241 (7 actives)


In [4]:
from sklearn.ensemble import RandomForestClassifier

def cv_predict(X, y, groups, n_splits=5, seed=mu.SEED, shuffle_labels=False):
    """Out-of-fold probability of being active, using a scaffold-aware split.

    Every compound is predicted by a model that never saw its scaffold during
    training. Set shuffle_labels=True for the negative control.
    """
    y_used = y.copy()
    if shuffle_labels:
        y_used = np.random.default_rng(seed).permutation(y_used)

    oof = np.zeros(len(y_used), dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in cv.split(X, y_used, groups):
        rf = RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                    random_state=seed, n_jobs=-1)
        rf.fit(X[train_idx], y_used[train_idx])
        oof[test_idx] = rf.predict_proba(X[test_idx])[:, 1]

    return oof, y_used

In [5]:
oof_morgan, _ = cv_predict(X_morgan, y, groups)
oof_ifp, _ = cv_predict(X_ifp, y, groups)
print("out-of-fold predictions done")

out-of-fold predictions done


In [6]:
rows = [
    mu.metric_row("Vina score",           y, -comp["score"].to_numpy()),
    mu.metric_row("Heavy atoms only",     y,  comp["heavy"].to_numpy()),
    mu.metric_row("RF on Morgan FP",      y,  oof_morgan),
    mu.metric_row("RF on interaction FP", y,  oof_ifp),
]
table = pd.DataFrame(rows)
print(table.to_string(index=False))

              method   AUC  CI_low  CI_high  EF_1% hits_top_1%  EF_5% hits_top_5%
          Vina score 0.709   0.591    0.818   15.2        6/12    7.5       15/61
    Heavy atoms only 0.576   0.470    0.680    5.1        2/12    3.0        6/61
     RF on Morgan FP 0.875   0.780    0.952   30.5       12/12   16.5       33/61
RF on interaction FP 0.794   0.687    0.888   25.4       10/12   11.5       23/61


In [7]:
for name, X in [("Morgan", X_morgan), ("interaction FP", X_ifp)]:
    oof_shuffled, y_shuffled = cv_predict(X, y, groups, shuffle_labels=True)
    auc, lo, hi = mu.roc_auc_ci(y_shuffled, oof_shuffled)
    print(f"label-shuffled {name:16s} AUC {auc:.3f}   (95% CI {lo:.3f} to {hi:.3f})")

label-shuffled Morgan           AUC 0.559   (95% CI 0.464 to 0.655)
label-shuffled interaction FP   AUC 0.636   (95% CI 0.551 to 0.719)


In [8]:
from sklearn.metrics import roc_auc_score

real = {"Morgan": 0.875, "interaction FP": 0.794}

for name, X in [("Morgan", X_morgan), ("interaction FP", X_ifp)]:
    aucs = []
    for s in range(10):
        oof_s, y_s = cv_predict(X, y, groups, seed=1000 + s, shuffle_labels=True)
        aucs.append(roc_auc_score(y_s, oof_s))
    aucs = np.array(aucs)
    beats = int((aucs >= real[name]).sum())
    print(f"{name:16s} shuffled AUC: mean {aucs.mean():.3f}, "
          f"range {aucs.min():.3f} to {aucs.max():.3f}, "
          f"{beats}/10 reached the real AUC of {real[name]:.3f}")

Morgan           shuffled AUC: mean 0.490, range 0.390 to 0.580, 0/10 reached the real AUC of 0.875
interaction FP   shuffled AUC: mean 0.499, range 0.424 to 0.579, 0/10 reached the real AUC of 0.794


In [9]:
table.to_csv("../results/ml/ml_comparison.csv", index=False)

pd.DataFrame({
    "id": comp["id"],
    "label": y,
    "vina_score": comp["score"],
    "heavy": comp["heavy"],
    "rf_morgan": oof_morgan,
    "rf_ifp": oof_ifp,
}).to_csv("../results/ml/ml_predictions.csv", index=False)

print("saved ml_comparison.csv and ml_predictions.csv")

saved ml_comparison.csv and ml_predictions.csv
